This is a companion notebook for the book [Deep Learning with Python, Third Edition](https://www.manning.com/books/deep-learning-with-python-third-edition). For readability, it only contains runnable code blocks and section titles, and omits everything else in the book: text paragraphs, figures, and pseudocode.

**If you want to be able to follow what's going on, I recommend reading the notebook side by side with your copy of the book.**

The book's contents are available online at [deeplearningwithpython.io](https://deeplearningwithpython.io).

In [ ]:
!pip install torch --upgrade -q

## Timeseries forecasting

### Different kinds of timeseries tasks

### A temperature forecasting example

In [0]:
!wget https://s3.amazonaws.com/keras-datasets/jena_climate_2009_2016.csv.zip
!unzip jena_climate_2009_2016.csv.zip

In [0]:
import os

fname = os.path.join("jena_climate_2009_2016.csv")

with open(fname) as f:
    data = f.read()

lines = data.split("\n")
header = lines[0].split(",")
lines = lines[1:]
print(header)
print(len(lines))

In [0]:
import numpy as np

temperature = np.zeros((len(lines),))
raw_data = np.zeros((len(lines), len(header) - 1))

for i, line in enumerate(lines):
    values = [float(x) for x in line.split(",")[1:]]
    temperature[i] = values[1]
    raw_data[i, :] = values[:]

In [0]:
from matplotlib import pyplot as plt

plt.plot(range(len(temperature)), temperature)

In [0]:
plt.plot(range(1440), temperature[:1440])

In [0]:
num_train_samples = int(0.5 * len(raw_data))
num_val_samples = int(0.25 * len(raw_data))
num_test_samples = len(raw_data) - num_train_samples - num_val_samples
print("num_train_samples:", num_train_samples)
print("num_val_samples:", num_val_samples)
print("num_test_samples:", num_test_samples)

#### Preparing the data

In [0]:
mean = raw_data[:num_train_samples].mean(axis=0)
raw_data -= mean
std = raw_data[:num_train_samples].std(axis=0)
raw_data /= std

In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

# PyTorch: there's no keras.utils.timeseries_dataset_from_array, so we build a
# small Dataset that mirrors it: each item is a window of length
# `sequence_length` (taking every `sampling_rate`-th step) paired with the
# target aligned to the window's start index. A DataLoader then batches/shuffles.
class TimeseriesDataset(Dataset):
    def __init__(self, data, targets, sequence_length,
                 sampling_rate=1, start_index=0, end_index=None):
        self.data = np.asarray(data)
        self.targets = np.asarray(targets)
        self.sequence_length = sequence_length
        self.sampling_rate = sampling_rate
        # span covered by one window in raw timesteps
        span = (sequence_length - 1) * sampling_rate + 1
        if end_index is None:
            end_index = len(self.data)
        # valid window start positions (mirrors Keras' indexing)
        self.start_positions = np.arange(start_index, end_index - span + 1)

    def __len__(self):
        return len(self.start_positions)

    def __getitem__(self, i):
        start = self.start_positions[i]
        stop = start + (self.sequence_length - 1) * self.sampling_rate + 1
        window = self.data[start:stop:self.sampling_rate]
        target = self.targets[start]
        return (torch.tensor(window, dtype=torch.float32),
                torch.tensor(target, dtype=torch.float32))

int_sequence = np.arange(10)
dummy_dataset = TimeseriesDataset(
    data=int_sequence[:-3],
    targets=int_sequence[3:],
    sequence_length=3,
)
dummy_loader = DataLoader(dummy_dataset, batch_size=2)

for inputs, targets in dummy_loader:
    for i in range(inputs.shape[0]):
        print([int(x) for x in inputs[i]], int(targets[i]))

In [ ]:
sampling_rate = 6
sequence_length = 120
delay = sampling_rate * (sequence_length + 24 - 1)
batch_size = 256

# PyTorch: same windowing parameters as the book; start_index/end_index carve out
# the train/val/test splits, and shuffling is done by the DataLoader.
train_data = TimeseriesDataset(
    raw_data[:-delay],
    targets=temperature[delay:],
    sampling_rate=sampling_rate,
    sequence_length=sequence_length,
    start_index=0,
    end_index=num_train_samples,
)
train_dataset = DataLoader(train_data, batch_size=batch_size, shuffle=True)

val_data = TimeseriesDataset(
    raw_data[:-delay],
    targets=temperature[delay:],
    sampling_rate=sampling_rate,
    sequence_length=sequence_length,
    start_index=num_train_samples,
    end_index=num_train_samples + num_val_samples,
)
val_dataset = DataLoader(val_data, batch_size=batch_size, shuffle=True)

test_data = TimeseriesDataset(
    raw_data[:-delay],
    targets=temperature[delay:],
    sampling_rate=sampling_rate,
    sequence_length=sequence_length,
    start_index=num_train_samples + num_val_samples,
)
test_dataset = DataLoader(test_data, batch_size=batch_size, shuffle=True)

In [0]:
for samples, targets in train_dataset:
    print("samples shape:", samples.shape)
    print("targets shape:", targets.shape)
    break

#### A commonsense, non-machine-learning baseline

In [ ]:
def evaluate_naive_method(dataset):
    total_abs_err = 0.0
    samples_seen = 0
    for samples, targets in dataset:
        # PyTorch: samples/targets are torch tensors here; .numpy() to keep the
        # rest of the arithmetic identical to the book.
        samples, targets = samples.numpy(), targets.numpy()
        preds = samples[:, -1, 1] * std[1] + mean[1]
        total_abs_err += np.sum(np.abs(preds - targets))
        samples_seen += samples.shape[0]
    return total_abs_err / samples_seen

print(f"Validation MAE: {evaluate_naive_method(val_dataset):.2f}")
print(f"Test MAE: {evaluate_naive_method(test_dataset):.2f}")

#### Let's try a basic machine learning model

In [ ]:
import torch
from torch import nn

# PyTorch: shared training helpers reused by every model below. fit() runs the
# manual training/validation loop (tracking MSE loss + MAE like Keras' metrics)
# and keeps the best-by-val-MAE weights, mirroring ModelCheckpoint(save_best_only).
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
loss_fn = nn.MSELoss()
mae_fn = nn.L1Loss()

def evaluate(model, dataset):
    model.eval()
    total_mae, samples_seen = 0.0, 0
    with torch.no_grad():
        for samples, targets in dataset:
            samples, targets = samples.to(device), targets.to(device)
            preds = model(samples).squeeze(-1)
            total_mae += mae_fn(preds, targets).item() * samples.shape[0]
            samples_seen += samples.shape[0]
    return total_mae / samples_seen

def fit(model, train_dataset, val_dataset, epochs, checkpoint_path):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters())
    history = {"mae": [], "val_mae": []}
    best_val_mae = float("inf")
    for epoch in range(epochs):
        model.train()
        total_mae, samples_seen = 0.0, 0
        for samples, targets in train_dataset:
            samples, targets = samples.to(device), targets.to(device)
            preds = model(samples).squeeze(-1)
            loss = loss_fn(preds, targets)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_mae += mae_fn(preds, targets).item() * samples.shape[0]
            samples_seen += samples.shape[0]
        train_mae = total_mae / samples_seen
        val_mae = evaluate(model, val_dataset)
        history["mae"].append(train_mae)
        history["val_mae"].append(val_mae)
        if val_mae < best_val_mae:
            best_val_mae = val_mae
            torch.save(model.state_dict(), checkpoint_path)
        print(f"Epoch {epoch + 1}: mae {train_mae:.4f} - val_mae {val_mae:.4f}")
    return history

# PyTorch: a functional Keras model becomes an nn.Module subclass. Flatten the
# (sequence_length, num_features) window before the Dense/Linear stack.
class DenseModel(nn.Module):
    def __init__(self, sequence_length, num_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(sequence_length * num_features, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
        )

    def forward(self, x):
        return self.net(x)

model = DenseModel(sequence_length, raw_data.shape[-1])
history = fit(model, train_dataset, val_dataset, epochs=10,
              checkpoint_path="jena_dense.pt")

# PyTorch: reload the best weights via state_dict (no whole-model .keras file).
model.load_state_dict(torch.load("jena_dense.pt"))
print(f"Test MAE: {evaluate(model, test_dataset):.2f}")

In [0]:
import matplotlib.pyplot as plt

loss = history.history["mae"]
val_loss = history.history["val_mae"]
epochs = range(1, len(loss) + 1)
plt.figure()
plt.plot(epochs, loss, "r--", label="Training MAE")
plt.plot(epochs, val_loss, "b", label="Validation MAE")
plt.title("Training and validation MAE")
plt.legend()
plt.show()

#### Let's try a 1D convolutional model

In [ ]:
# PyTorch: nn.Conv1d expects (batch, channels, length), i.e. channels-first,
# whereas Keras Conv1D uses (batch, length, channels). We permute the input
# window to channels-first, and use mean over the time axis for global avg pool.
class ConvModel(nn.Module):
    def __init__(self, num_features):
        super().__init__()
        self.conv1 = nn.Conv1d(num_features, 8, kernel_size=24)
        self.pool1 = nn.MaxPool1d(2)
        self.conv2 = nn.Conv1d(8, 8, kernel_size=12)
        self.pool2 = nn.MaxPool1d(2)
        self.conv3 = nn.Conv1d(8, 8, kernel_size=6)
        self.fc = nn.Linear(8, 1)

    def forward(self, x):
        x = x.permute(0, 2, 1)  # (batch, length, features) -> (batch, features, length)
        x = torch.relu(self.conv1(x))
        x = self.pool1(x)
        x = torch.relu(self.conv2(x))
        x = self.pool2(x)
        x = torch.relu(self.conv3(x))
        x = x.mean(dim=2)  # global average pooling over the time axis
        return self.fc(x)

model = ConvModel(raw_data.shape[-1])
history = fit(model, train_dataset, val_dataset, epochs=10,
              checkpoint_path="jena_conv.pt")

model.load_state_dict(torch.load("jena_conv.pt"))
print(f"Test MAE: {evaluate(model, test_dataset):.2f}")

### Recurrent neural networks

In [ ]:
# PyTorch: nn.LSTM with batch_first=True returns (output, (h_n, c_n)); we take
# the last timestep's output (out[:, -1, :]) to feed the Dense/Linear head,
# which is what Keras' LSTM(16) (return_sequences=False) produces.
class LSTMModel(nn.Module):
    def __init__(self, num_features):
        super().__init__()
        self.lstm = nn.LSTM(num_features, 16, batch_first=True)
        self.fc = nn.Linear(16, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

model = LSTMModel(raw_data.shape[-1])
history = fit(model, train_dataset, val_dataset, epochs=10,
              checkpoint_path="jena_lstm.pt")

model.load_state_dict(torch.load("jena_lstm.pt"))
print(f"Test MAE: {evaluate(model, test_dataset):.2f}")

#### Understanding recurrent neural networks

In [0]:
import numpy as np

timesteps = 100
input_features = 32
output_features = 64
inputs = np.random.random((timesteps, input_features))
state_t = np.zeros((output_features,))
W = np.random.random((output_features, input_features))
U = np.random.random((output_features, output_features))
b = np.random.random((output_features,))
successive_outputs = []
for input_t in inputs:
    output_t = np.tanh(np.dot(W, input_t) + np.dot(U, state_t) + b)
    successive_outputs.append(output_t)
    state_t = output_t
final_output_sequence = np.concatenate(successive_outputs, axis=0)

#### A recurrent layer in Keras

In [ ]:
num_features = 14
# PyTorch: layers.SimpleRNN -> nn.RNN(batch_first=True). By default nn.RNN
# returns the full sequence; we read the last timestep for the "last output only"
# behavior. No fixed sequence length is needed (works on variable lengths).
rnn = nn.RNN(num_features, 16, batch_first=True)

In [ ]:
num_features = 14
steps = 120
rnn = nn.RNN(num_features, 16, batch_first=True)
dummy = torch.zeros((1, steps, num_features))
out, _ = rnn(dummy)
# PyTorch: "return_sequences=False" == taking only the last timestep.
outputs = out[:, -1, :]
print(outputs.shape)

In [ ]:
num_features = 14
steps = 120
rnn = nn.RNN(num_features, 16, batch_first=True)
dummy = torch.zeros((1, steps, num_features))
# PyTorch: "return_sequences=True" == keeping the full output sequence.
outputs, _ = rnn(dummy)
print(outputs.shape)

In [ ]:
# PyTorch: stacking SimpleRNNs with return_sequences=True is just nn.RNN with
# num_layers=3 (each layer feeds its full sequence to the next); we then take the
# last timestep of the final layer's output, matching the book's last RNN(16).
rnn = nn.RNN(num_features, 16, num_layers=3, batch_first=True)
dummy = torch.zeros((1, steps, num_features))
out, _ = rnn(dummy)
outputs = out[:, -1, :]

#### Getting the most out of recurrent neural networks

#### Using recurrent dropout to fight overfitting

In [ ]:
# PyTorch: nn.LSTM has no per-timestep `recurrent_dropout`; its `dropout` arg
# only applies between stacked layers (and needs num_layers > 1). For a single
# LSTM layer we approximate the book's regularization with the dropout that
# follows the recurrent layer (layers.Dropout(0.5)).
class LSTMDropoutModel(nn.Module):
    def __init__(self, num_features):
        super().__init__()
        self.lstm = nn.LSTM(num_features, 32, batch_first=True)
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(32, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        x = self.dropout(out[:, -1, :])
        return self.fc(x)

model = LSTMDropoutModel(raw_data.shape[-1])
history = fit(model, train_dataset, val_dataset, epochs=50,
              checkpoint_path="jena_lstm_dropout.pt")

#### Stacking recurrent layers

In [ ]:
# PyTorch: two stacked GRUs == nn.GRU(num_layers=2). The `dropout` arg applies
# between the two layers (PyTorch has no `recurrent_dropout`), and we add a final
# nn.Dropout(0.5) before the head to match the book's layers.Dropout(0.5).
class StackedGRUModel(nn.Module):
    def __init__(self, num_features):
        super().__init__()
        self.gru = nn.GRU(num_features, 32, num_layers=2,
                          batch_first=True, dropout=0.5)
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(32, 1)

    def forward(self, x):
        out, _ = self.gru(x)
        x = self.dropout(out[:, -1, :])
        return self.fc(x)

model = StackedGRUModel(raw_data.shape[-1])
history = fit(model, train_dataset, val_dataset, epochs=50,
              checkpoint_path="jena_stacked_gru_dropout.pt")
model.load_state_dict(torch.load("jena_stacked_gru_dropout.pt"))
print(f"Test MAE: {evaluate(model, test_dataset):.2f}")

#### Using bidirectional RNNs

In [ ]:
# PyTorch: layers.Bidirectional(LSTM(16)) -> nn.LSTM(bidirectional=True). The
# output's last dim is 2*hidden, and we concatenate the final forward state with
# the final backward state (the latter is read at the first timestep, out[:, 0,
# hidden:]) to mirror Keras' bidirectional "last output".
class BiLSTMModel(nn.Module):
    def __init__(self, num_features, hidden=16):
        super().__init__()
        self.hidden = hidden
        self.lstm = nn.LSTM(num_features, hidden, batch_first=True,
                            bidirectional=True)
        self.fc = nn.Linear(2 * hidden, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        forward_last = out[:, -1, :self.hidden]
        backward_last = out[:, 0, self.hidden:]
        return self.fc(torch.cat([forward_last, backward_last], dim=1))

model = BiLSTMModel(raw_data.shape[-1])
history = fit(model, train_dataset, val_dataset, epochs=10,
              checkpoint_path="jena_bidir_lstm.pt")

### Going even further